# IHES Cube — Symmetry + Reverse MLP Beam Search

This notebook validates the 48 IHES relabellings, searches selected symmetry frames in both direct and reverse projections, converts every candidate back to original coordinates, and accepts only exact replay-valid paths. It uses every official generator and never relies on the row order of the reverse-neighbour tensor.


In [ ]:
from pathlib import Path
PUZZLE_ID = 106
MODEL_ID = "1778521793"
BEAM_WIDTH = 1_000_000
MAX_DEPTH = 30
PARENT_CHUNK = 250_000
INFERENCE_BATCH = 8_192
DEVICE = "cuda"
K_SYM = 2
SYMMETRY_SEED = 0
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/cheldieva-l/ihes-dual-model-beam-search'
REPOSITORY_REF = "main"
CHECKOUT = Path("/kaggle/working/ihes-dual-model-beam-search")
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(CHECKOUT)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT), "--no-deps", "-q"], check=True)
sys.path.insert(0, str(CHECKOUT))
print(subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import numpy as np
import torch

from ihes_dual.assets import find_competition_assets
from ihes_dual.model import load_mlp2rb
from ihes_dual.puzzle import IHESPuzzle, load_test_state
from ihes_dual.registry import resolve_model

assets = find_competition_assets("/kaggle/input")
puzzle = IHESPuzzle.from_puzzle_info(assets.puzzle_info)
start = load_test_state(assets.test_csv, PUZZLE_ID)
model_spec = resolve_model("/kaggle/input", MODEL_ID)
if model_spec.model_id != MODEL_ID:
    raise AssertionError("the resolved checkpoint is not tied to the configured model ID")
model = load_mlp2rb(model_spec, DEVICE)
print({
    "puzzle_id": PUZZLE_ID,
    "model_id": model_spec.model_id,
    "checkpoint": model_spec.checkpoint.name,
    "epoch": model_spec.epoch,
    "beam_width": BEAM_WIDTH,
    "device": DEVICE,
    "generator_count": puzzle.generator_count,
})
assert puzzle.generator_count == 18


In [ ]:
from ihes_dual.assets import find_symmetry_file
from ihes_dual.symmetry import load_symmetry_frames

symmetry_path = find_symmetry_file("/kaggle/input")
frames_all = load_symmetry_frames(str(symmetry_path), puzzle)
identity_index = next(
    index for index, frame in enumerate(frames_all)
    if np.array_equal(frame.rotation, np.arange(puzzle.state_size))
)
rng = np.random.default_rng(SYMMETRY_SEED)
other_indices = [index for index in range(len(frames_all)) if index != identity_index]
chosen_indices = [identity_index]
if K_SYM > 1:
    chosen_indices.extend(rng.choice(other_indices, size=K_SYM - 1, replace=False).tolist())
frames = [frames_all[index] for index in chosen_indices]
print("validated symmetries:", len(frames_all), "chosen:", chosen_indices)


In [ ]:
from dataclasses import asdict
from ihes_dual.beam import BeamConfig
from ihes_dual.solve import solve_symmetry_reverse, write_run_log
from ihes_dual.submission import build_submission, validate_submission

config = BeamConfig(
    beam_width=BEAM_WIDTH,
    max_depth=MAX_DEPTH,
    parent_chunk=PARENT_CHUNK,
    inference_batch=INFERENCE_BATCH,
    device=DEVICE,
    autocast=True,
    prune_immediate_inverse=False,
    diagnostic_protect=False,
)
best, runs = solve_symmetry_reverse(
    puzzle,
    start,
    model,
    config,
    frames,
    model_id=MODEL_ID,
    include_direct=True,
    include_reverse=True,
)
if best is None:
    raise RuntimeError("no replay-valid direct or reverse candidate was found")
if not puzzle.verify_solution(start, best.path):
    raise AssertionError("best transformed path failed exact original-coordinate replay")
print("best:", best)
print("solution:", puzzle.encode_path(best.path))

run_dir = OUTPUT_DIR / "runs" / f"model-{MODEL_ID}" / f"puzzle-{PUZZLE_ID}" / "symmetry-reverse"
write_run_log(
    run_dir / "run.json",
    {
        "mode": "symmetry-reverse",
        "model_id": MODEL_ID,
        "puzzle_id": PUZZLE_ID,
        "chosen_absolute_symmetry_indices": chosen_indices,
        "best": asdict(best),
        "best_path": puzzle.encode_path(best.path),
        "runs": runs,
        "config": asdict(config),
    },
)
submission_path = build_submission(
    assets.sample_submission,
    OUTPUT_DIR / "submission.csv",
    puzzle,
    {PUZZLE_ID: best.path},
)
print("validation:", validate_submission(submission_path, assets.test_csv, puzzle))
